# Production Training — ELECTRA (Single Model Run)

**Purpose:** Fine-tune `google/electra-base-discriminator` for up to 5 epochs with early stopping, producing a deployable artifact in `model_registry/staging/`.

**Usage:** Run all cells top-to-bottom. No model selection needed — ELECTRA is the only model trained in this notebook.

**Model:**

| Key | Checkpoint | bs | accum | Effective batch |
|---|---|---|---|---|
| `"electra"` | `google/electra-base-discriminator` | 32 | 4 | 128 |

**Hardware target:** RTX 3060 Laptop, 6 GB VRAM.

**Features:**
- BF16 mixed precision (Ampere native, no GradScaler)
- TF32 tensor-core throughput for FP32 ops
- Offline pre-tokenization — all splits tokenized once
- Cosine LR schedule with warmup ratio 6% (ELECTRA benefits from longer warmup)
- Learning rate 2e-5
- Effective batch size 128 (comparable training dynamics to previous models)
- Early stopping on `val_macro_f1` (patience=3)
- Gradient norm logging + NaN/Inf guard
- Per-class F1 logged every epoch (Code Injection monitored in epoch summary line)
- VRAM preflight check before training
- Alpha-balanced focal loss (γ=2) — addresses Code Injection precision vs recall tradeoff
- Artifact persistence to `model_registry/staging/`

In [ ]:
import torch
import transformers

print(f"PyTorch     : {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "CUDA not available — check PyTorch installation"

In [ ]:
import os
import sys
import json
import math
import time
import random
import subprocess
import shutil
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast                     # no GradScaler needed with BF16

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,               # replaces linear schedule
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# TF32 — Ampere tensor-core matmul throughput for FP32 ops outside autocast
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
# Deterministic mode — no autotuner thrashing on variable-length sequences
torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
print(f"Platform : {sys.platform}")
print(f"TF32 enabled: matmul={torch.backends.cuda.matmul.allow_tf32}  "
      f"cudnn={torch.backends.cudnn.allow_tf32}")
print("Imports OK.")

In [ ]:
# ── ELECTRA model configuration ─────────────────────────────────────────────────
# Only ELECTRA is trained in this notebook.
ACTIVE_MODEL = "electra"

MODEL_REGISTRY = {
    "electra": {
        "model_id":                    "google/electra-base-discriminator",
        "per_device_train_batch_size": 32,    # matches bert-base footprint; effective batch = 128
        "gradient_accumulation_steps": 4,
        "learning_rate":               2e-5,  # ELECTRA discriminator fine-tuning sweet spot
        "warmup_ratio":                0.06,  # longer warmup stabilises ELECTRA discriminator head
    },
}

assert ACTIVE_MODEL in MODEL_REGISTRY, (
    f"Unknown model: {ACTIVE_MODEL}. Only 'electra' is supported in this notebook."
)
_sel = MODEL_REGISTRY[ACTIVE_MODEL]

TRAINING_CONFIG = {
    "model_key":                   ACTIVE_MODEL,
    "model_id":                    _sel["model_id"],
    "dataset_version":             "v3_907k_cleaned",
    "per_device_train_batch_size": _sel["per_device_train_batch_size"],
    "gradient_accumulation_steps": _sel["gradient_accumulation_steps"],
    "num_train_epochs":            5,
    "early_stop_patience":         3,
    "learning_rate":               _sel["learning_rate"],
    "weight_decay":                0.01,   # AdamW — applied only to non-bias/LayerNorm params (see build_optimizer)
    "max_grad_norm":               1.0,
    "lr_schedule":                 "cosine",
    "warmup_ratio":                _sel["warmup_ratio"],
    "bf16":                        True,
    "max_seq_len":                 128,
    "log_every_steps":             200,
    # Class weight tuning for Code Injection (3.5% minority class):
    # ci_weight_scale=1.0 uses raw inverse-frequency weights.
    # Reduce toward 0.5 if Code Injection precision is too low (false alarms).
    # Increase toward 2.0 if Code Injection recall drops (missed attacks).
    "ci_weight_scale":             0.6,
    # Focal loss gamma — down-weights easy examples, focuses on hard misclassifications.
    # gamma=0 reduces to weighted cross-entropy; gamma=2 is the standard choice (Lin et al. 2017).
    "focal_gamma":                 2.0,
}

accum_steps = TRAINING_CONFIG["gradient_accumulation_steps"]
eff_bs      = TRAINING_CONFIG["per_device_train_batch_size"] * accum_steps

print(f"Model    : {TRAINING_CONFIG['model_id']} ({ACTIVE_MODEL})")
print(f"Batch    : {TRAINING_CONFIG['per_device_train_batch_size']} x {accum_steps} = {eff_bs} effective")
print(f"Epochs   : {TRAINING_CONFIG['num_train_epochs']}  (patience={TRAINING_CONFIG['early_stop_patience']})")
print(f"LR       : {TRAINING_CONFIG['learning_rate']}  warmup={TRAINING_CONFIG['warmup_ratio']*100:.0f}%  schedule={TRAINING_CONFIG['lr_schedule']}")
print(f"BF16     : {TRAINING_CONFIG['bf16']}  |  max_seq_len={TRAINING_CONFIG['max_seq_len']}")
print(f"Loss     : FocalLoss(gamma={TRAINING_CONFIG['focal_gamma']})  |  ci_weight_scale={TRAINING_CONFIG['ci_weight_scale']}")
print(f"Device   : {DEVICE}")

In [ ]:
DATA_DIR  = os.path.join("..", "data", "processed", "v3_907k_cleaned")
TEXT_COL  = "combined_payload"
LABEL_COL = "final_label"

df_train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
df_val   = pd.read_parquet(os.path.join(DATA_DIR, "validation.parquet"))
df_test  = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

print(f"Train      : {len(df_train):,}")
print(f"Validation : {len(df_val):,}")
print(f"Test       : {len(df_test):,}")

# Column assertions — fail fast if schema changed
assert TEXT_COL  in df_train.columns, f"Missing column: {TEXT_COL}"
assert LABEL_COL in df_train.columns, f"Missing column: {LABEL_COL}"
assert TEXT_COL  in df_val.columns,   f"Missing column in val: {TEXT_COL}"
assert LABEL_COL in df_val.columns,   f"Missing column in val: {LABEL_COL}"
assert TEXT_COL  in df_test.columns,  f"Missing column in test: {TEXT_COL}"
assert LABEL_COL in df_test.columns,  f"Missing column in test: {LABEL_COL}"

print(f"\nLabel distribution (train):")
print(df_train[LABEL_COL].value_counts().to_string())

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(df_train[LABEL_COL])

NUM_CLASSES = len(label_encoder.classes_)
LABEL_NAMES = list(label_encoder.classes_)

# Guard: all 4 classes must be present in train split.
# If any class is missing, transform() on val/test raises ValueError silently downstream.
EXPECTED_CLASSES = {"Code Injection", "Normal", "Other Attacks", "SQL Injection"}
assert set(LABEL_NAMES) == EXPECTED_CLASSES, (
    f"Label mismatch. Found: {set(LABEL_NAMES)}\nExpected: {EXPECTED_CLASSES}"
)
assert NUM_CLASSES == 4, f"Expected 4 classes, got {NUM_CLASSES}"

df_train["label_id"] = label_encoder.transform(df_train[LABEL_COL])
df_val["label_id"]   = label_encoder.transform(df_val[LABEL_COL])
df_test["label_id"]  = label_encoder.transform(df_test[LABEL_COL])

print(f"Classes ({NUM_CLASSES}):")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {i} -> {name}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG["model_id"])

sample = df_train[TEXT_COL].iloc[0][:200]
enc    = tokenizer(sample, max_length=TRAINING_CONFIG["max_seq_len"], truncation=True)
print(f"Tokenizer loaded   : {TRAINING_CONFIG['model_id']}")
print(f"Sample token count : {len(enc['input_ids'])}")
print(f"Keys               : {list(enc.keys())}")

In [ ]:
# ── VRAM preflight (forward + backward at configured bs) ───────────────────────
HEADROOM_MB = 256  # reserve a small safety margin for driver/CuDNN and other processes
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    PREFLIGHT_BS = TRAINING_CONFIG["per_device_train_batch_size"]
    _seq         = TRAINING_CONFIG["max_seq_len"]

    total_gpu_mb = torch.cuda.get_device_properties(0).total_memory / 1024**2
    VRAM_PASS_MB = int(total_gpu_mb - HEADROOM_MB)

    _m   = AutoModelForSequenceClassification.from_pretrained(
        TRAINING_CONFIG["model_id"], num_labels=NUM_CLASSES
    ).to(DEVICE).train()
    _ids = torch.randint(0, 1000, (PREFLIGHT_BS, _seq), device=DEVICE)
    _msk = torch.ones(PREFLIGHT_BS, _seq, dtype=torch.long, device=DEVICE)
    _lbl = torch.zeros(PREFLIGHT_BS, dtype=torch.long, device=DEVICE)

    with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
        _out = _m(input_ids=_ids, attention_mask=_msk, labels=_lbl)
    _out.loss.backward()

    peak_mb = torch.cuda.max_memory_allocated() / 1024**2

    if peak_mb > VRAM_PASS_MB:
        raise RuntimeError(
            f"VRAM preflight FAILED: {peak_mb:.0f} MB > {VRAM_PASS_MB} MB threshold. "
            f"Reduce per_device_train_batch_size in TRAINING_CONFIG or increase HEADROOM_MB."
        )
    elif peak_mb < VRAM_PASS_MB * 0.77:
        print(
            f"[PASS] Peak VRAM = {peak_mb:.0f} MB at bs={PREFLIGHT_BS} — "
            f"well under budget ({VRAM_PASS_MB:.0f} MB). Consider increasing per_device_train_batch_size."
        )
    else:
        print(f"[PASS] VRAM preflight: {peak_mb:.0f} MB at bs={PREFLIGHT_BS} (budget {VRAM_PASS_MB:.0f} MB)")

    del _m, _ids, _msk, _lbl, _out
    torch.cuda.empty_cache()
else:
    print("No CUDA — skipping VRAM preflight.")
    peak_mb = 0.0

In [ ]:
# ── Offline pre-tokenization ──────────────────────────────────────────────────
def _preprocess(df_split, tok, max_len):
    """Batch-tokenize an entire DataFrame split in one call."""
    encoded = tok(
        list(df_split[TEXT_COL]),
        truncation=True,
        max_length=max_len,
        padding=False,      # DataCollatorWithPadding handles per-batch padding
    )
    return {
        "input_ids":      encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }


class WAFDataset(Dataset):
    """Pre-tokenized dataset — no tokenizer call in __getitem__."""

    def __init__(self, precomputed, labels):
        self.input_ids      = precomputed["input_ids"]
        self.attention_mask = precomputed["attention_mask"]
        self.labels         = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.input_ids[idx],     dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels":         torch.tensor(int(self.labels[idx]),    dtype=torch.long),
        }


print("Pre-tokenizing splits ... (one-time cost, ~10-30 s)")
_t0 = time.time()
precomputed_train = _preprocess(df_train, tokenizer, TRAINING_CONFIG["max_seq_len"])
precomputed_val   = _preprocess(df_val,   tokenizer, TRAINING_CONFIG["max_seq_len"])
precomputed_test  = _preprocess(df_test,  tokenizer, TRAINING_CONFIG["max_seq_len"])
print(f"Done in {time.time() - _t0:.1f} s")
print(f"  Train : {len(precomputed_train['input_ids']):,}")
print(f"  Val   : {len(precomputed_val['input_ids']):,}")
print(f"  Test  : {len(precomputed_test['input_ids']):,}")
print("WAFDataset (offline) defined.")

In [ ]:
def build_dataloaders(tok, batch_size):
    """
    Build train/val/test DataLoaders from pre-tokenized splits.
    pin_memory=True   — page-locked tensors enable async H2D with non_blocking=True
    num_workers=0     — required on Windows to prevent DataLoader spawn deadlock
    generator         — seeded for reproducible shuffle order
    eval_batch_size   — 2x train bs for val/test (no backward pass, fits more in VRAM)
    """
    collator = DataCollatorWithPadding(tokenizer=tok, padding=True)
    common   = dict(collate_fn=collator, num_workers=0, pin_memory=True)
    _gen     = torch.Generator().manual_seed(SEED)

    train_loader = DataLoader(
        WAFDataset(precomputed_train, df_train["label_id"]),
        batch_size=batch_size, shuffle=True, generator=_gen, **common,
    )
    eval_bs = batch_size * 2  # no backward pass — can fit 2x without VRAM pressure
    val_loader = DataLoader(
        WAFDataset(precomputed_val, df_val["label_id"]),
        batch_size=eval_bs, shuffle=False, **common,
    )
    test_loader = DataLoader(
        WAFDataset(precomputed_test, df_test["label_id"]),
        batch_size=eval_bs, shuffle=False, **common,
    )
    return train_loader, val_loader, test_loader


bs = TRAINING_CONFIG["per_device_train_batch_size"]
train_loader, val_loader, test_loader = build_dataloaders(tokenizer, bs)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

_batch = next(iter(train_loader))
print(f"\nFirst batch input_ids shape : {_batch['input_ids'].shape}")
print(f"Dynamic pad-to length       : {_batch['input_ids'].shape[1]} tokens "
      f"(< {TRAINING_CONFIG['max_seq_len']} = dynamic padding working)")
del _batch

In [ ]:
# ── Class weights + FocalLoss ───────────────────────────────────────────────────
# Inverse-frequency weighting: w_c = N_total / (K * N_c)
# ci_weight_scale in TRAINING_CONFIG scales the Code Injection weight independently.
# <1.0 reduces false positives (precision up, recall down slightly)
# >1.0 reduces false negatives (recall up, precision down)
counts  = np.bincount(df_train["label_id"].values, minlength=NUM_CLASSES).astype(np.float64)
weights = counts.sum() / (NUM_CLASSES * counts)

# Apply ci_weight_scale to Code Injection class only
ci_idx   = LABEL_NAMES.index("Code Injection")
ci_scale = TRAINING_CONFIG.get("ci_weight_scale", 1.0)
weights[ci_idx] *= ci_scale

CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print(f"Class weights (inverse-frequency, ci_weight_scale={ci_scale}):")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name:25s}: {CLASS_WEIGHTS[i].item():.4f}")
print(f"  [Code Injection weight scaled by {ci_scale} — targeting precision/recall balance]")


# ── FocalLoss ────────────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Alpha-balanced focal loss (Lin et al., RetinaNet ICCV 2017).

    - alpha (CLASS_WEIGHTS): per-class inverse-frequency weights — same role as in weighted CE
    - gamma=2: down-weights easy examples (high p_t), focuses training on hard misclassifications
    - When gamma=0, reduces exactly to weighted cross-entropy

    Why focal loss here:
    Weighted CE was over-predicting Code Injection (precision=0.9393, recall=0.9988) because
    the model was heavily penalised for any missed CI. Focal adds a (1 - p_t)^gamma modulating
    factor that reduces loss from confidently-correct easy examples, so borderline hard examples
    (the CI/Normal confusion boundary) receive proportionally more gradient signal.
    """

    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)                          # probability of correct class
        return ((1.0 - pt) ** self.gamma * ce).mean()


print(f"FocalLoss defined  (gamma={TRAINING_CONFIG['focal_gamma']}, alpha=CLASS_WEIGHTS)")

In [ ]:
def build_optimizer(model, lr, weight_decay):
    no_decay = ["bias", "LayerNorm.weight"]
    return torch.optim.AdamW(
        [
            {"params": [p for n, p in model.named_parameters()
                        if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
            {"params": [p for n, p in model.named_parameters()
                        if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
        ],
        lr=lr,
    )


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion,
                    device, accum_steps, max_grad_norm, log_every=200):
    """
    BF16 training loop with gradient norm logging and NaN/Inf guard.
    - grad_norm captured from clip_grad_norm_ return value
    - NaN/Inf raises immediately — does not waste the rest of the epoch
    - No GradScaler: BF16 has FP32 dynamic range
    """
    model.train()
    total_loss = 0.0
    total_norm = 0.0
    norm_steps = 0
    optimizer.zero_grad()
    n_steps = len(dataloader)
    t0      = time.time()

    for step, batch in enumerate(dataloader):
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbls = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbls) / accum_steps

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Training diverged at step {step + 1}: loss={loss.item():.6f}. "
                "Check learning rate, gradient clipping, and data integrity."
            )

        loss.backward()
        total_loss += loss.item() * accum_steps

        if (step + 1) % accum_steps == 0 or (step + 1) == n_steps:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_norm += grad_norm.item()
            norm_steps += 1

        if (step + 1) % log_every == 0 or (step + 1) == n_steps:
            elapsed  = time.time() - t0
            eta      = elapsed / (step + 1) * (n_steps - step - 1)
            avg_loss = total_loss / (step + 1)
            avg_norm = total_norm / max(norm_steps, 1)
            pct      = (step + 1) / n_steps * 100
            vram     = (f"  VRAM {torch.cuda.max_memory_allocated()/1e6:.0f}MB"
                        if device.type == "cuda" else "")
            print(f"  step {step+1:>5,}/{n_steps:,} ({pct:4.1f}%)  "
                  f"loss {avg_loss:.4f}  grad_norm {avg_norm:.3f}  "
                  f"elapsed {elapsed:.0f}s  ETA {eta:.0f}s{vram}")

    return total_loss / n_steps


print("build_optimizer + train_one_epoch defined.")

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """
    Evaluate model on a DataLoader.
    Returns (avg_loss, accuracy, macro_f1, per_class_f1_dict, all_preds, all_labels).
    Code Injection F1 is always printed — first class to collapse at 3.5% prevalence.
    """
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in dataloader:
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbls = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbls)

        total_loss += loss.item()
        all_preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    avg_loss = total_loss / len(dataloader)
    acc      = accuracy_score(all_labels, all_preds)
    report   = classification_report(
        all_labels, all_preds,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
    macro_f1     = report["macro avg"]["f1-score"]
    per_class_f1 = {name: report[name]["f1-score"] for name in LABEL_NAMES}

    ci_f1 = per_class_f1.get("Code Injection", 0.0)
    print(f"  Code Injection F1 : {ci_f1:.4f}  (3.5% minority -> watch for collapse)")

    model.train()  # restore training mode — evaluate() must be state-neutral
    return avg_loss, acc, macro_f1, per_class_f1, all_preds, all_labels


print("evaluate defined.")

In [ ]:
cfg         = TRAINING_CONFIG
accum_steps = cfg["gradient_accumulation_steps"]
num_epochs  = cfg["num_train_epochs"]

model = AutoModelForSequenceClassification.from_pretrained(
    cfg["model_id"], num_labels=NUM_CLASSES
).to(DEVICE)

# torch.compile requires Triton (Inductor backend) — unavailable on Windows
_CAN_COMPILE = sys.platform != "win32" and DEVICE.type == "cuda"
if _CAN_COMPILE:
    torch._dynamo.config.suppress_errors = True
    model = torch.compile(model)
    print("torch.compile applied  (Inductor + Triton)")
else:
    print(f"torch.compile skipped — Triton unavailable on {sys.platform}; running eager mode")

steps_per_epoch = math.ceil(len(train_loader) / accum_steps)
total_steps     = steps_per_epoch * num_epochs
warmup_steps    = int(cfg["warmup_ratio"] * total_steps)

optimizer = build_optimizer(model, cfg["learning_rate"], cfg["weight_decay"])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = FocalLoss(alpha=CLASS_WEIGHTS, gamma=cfg.get("focal_gamma", 2.0))
# No GradScaler — BF16 has FP32 dynamic range; loss scaling unnecessary

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel       : {cfg['model_id']}")
print(f"Parameters  : {total_params:,} total  |  {trainable_params:,} trainable (100%)")
print(f"Steps/epoch : {steps_per_epoch:,}  |  Total: {total_steps:,}  |  Warmup: {warmup_steps} ({cfg['warmup_ratio']*100:.0f}%)")
print(f"Schedule    : cosine with {warmup_steps}-step warmup")
print(f"Criterion   : FocalLoss(gamma={cfg.get('focal_gamma', 2.0)}, alpha=CLASS_WEIGHTS)")
print(f"\nReady to train.")

In [ ]:
PATIENCE       = cfg["early_stop_patience"]
best_val_f1    = -1.0
epochs_no_imp  = 0
model_name     = cfg["model_key"]
best_ckpt_dir  = os.path.join("model_registry", "staging")
best_ckpt_path = os.path.join(best_ckpt_dir, f".best_{model_name}_ckpt.pt")
os.makedirs(best_ckpt_dir, exist_ok=True)

training_log = []

print(f"Starting training: {num_epochs} epochs, early-stop patience={PATIENCE}")
print(f"Best checkpoint  : {best_ckpt_path}")
print("=" * 70)

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion,
        DEVICE, accum_steps, cfg["max_grad_norm"],
        log_every=cfg["log_every_steps"],
    )

    val_loss, val_acc, val_macro_f1, per_class_f1, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )

    elapsed      = time.time() - t0
    peak_vram_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else 0.0

    ci_f1_epoch = per_class_f1.get("Code Injection", 0.0)
    log_entry = {
        "epoch":           epoch,
        "train_loss":      round(train_loss,   5),
        "val_loss":        round(val_loss,     5),
        "val_acc":         round(val_acc,      5),
        "val_macro_f1":    round(val_macro_f1, 5),
        "per_class_f1":    {k: round(v, 5) for k, v in per_class_f1.items()},
        "stopped_early":   False,
        "peak_vram_mb":    round(peak_vram_mb, 1),
        "elapsed_s":       round(elapsed, 1),
        "ci_weight_scale": TRAINING_CONFIG.get("ci_weight_scale", 1.0),
    }
    training_log.append(log_entry)

    print(
        f"\nEpoch {epoch}/{num_epochs} | "
        f"train={train_loss:.4f}  val={val_loss:.4f}  "
        f"acc={val_acc:.4f}  macro_F1={val_macro_f1:.4f}  "
        f"CI_F1={ci_f1_epoch:.4f}  "
        f"VRAM={peak_vram_mb:.0f}MB  {elapsed/60:.1f}min"
    )

    if val_macro_f1 > best_val_f1:
        best_val_f1   = val_macro_f1
        epochs_no_imp = 0
        torch.save(model.state_dict(), best_ckpt_path)
        print(f"  New best macro-F1={best_val_f1:.4f} — checkpoint saved.")
    else:
        epochs_no_imp += 1
        print(f"  No improvement. Patience: {epochs_no_imp}/{PATIENCE}")
        if epochs_no_imp >= PATIENCE:
            print(f"\nEarly stopping after epoch {epoch} (no improvement for {PATIENCE} epochs).")
            training_log[-1]["stopped_early"] = True
            break

print(f"\n{'=' * 70}")
print(f" Training complete.")
print(f" Best val macro-F1 : {best_val_f1:.4f}")
print(f" Checkpoint        : {best_ckpt_path}")
print(f"{'=' * 70}")

In [ ]:
# Reload best-epoch weights before test evaluation
print(f"Loading best checkpoint from: {best_ckpt_path}")
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE, weights_only=True))

test_loss, test_acc, test_macro_f1, test_per_class_f1, test_preds, test_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

print(f"\n{'=' * 65}")
print(f" {ACTIVE_MODEL} — Test Set Results (best checkpoint)")
print(f"{'=' * 65}")
print(f" Test Loss     : {test_loss:.4f}")
print(f" Test Accuracy : {test_acc:.4f}")
print(f" Test Macro-F1 : {test_macro_f1:.4f}")
print(f"{'=' * 65}")
print()
print(classification_report(
    test_labels, test_preds,
    target_names=LABEL_NAMES,
    digits=4,
    zero_division=0,
))

assert test_acc > 0.50,       f"Test accuracy {test_acc:.4f} suspiciously low — check data pipeline"
assert not np.isnan(test_loss), "Test loss is NaN — check class weights or data"
print("Test evaluation complete.")

In [ ]:

# ── Persist run artifacts to model_registry/staging/ ───────────────────────────────────────
dataset_version = TRAINING_CONFIG["dataset_version"]
timestamp       = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir      = os.path.join("model_registry", "staging",
                               f"{model_name}_{dataset_version}_{timestamp}")
os.makedirs(output_dir, exist_ok=True)

# 1. config_used.json — exact training configuration at runtime
with open(os.path.join(output_dir, "config_used.json"), "w") as f:
    json.dump(TRAINING_CONFIG, f, indent=2)

# 2. training_log.json — per-epoch: loss, F1, per-class F1, VRAM, elapsed
with open(os.path.join(output_dir, "training_log.json"), "w") as f:
    json.dump(training_log, f, indent=2)

# 3. eval_report.json — per-class precision / recall / F1 on test set
eval_report_dict = classification_report(
    test_labels, test_preds,
    target_names=LABEL_NAMES,
    output_dict=True,
    zero_division=0,
)
with open(os.path.join(output_dir, "eval_report.json"), "w") as f:
    json.dump(eval_report_dict, f, indent=2)

# 4. git_hash.txt — commit SHA for full experiment provenance
try:
    git_hash = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        stderr=subprocess.DEVNULL,
    ).decode().strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    git_hash = "non-git-environment"
with open(os.path.join(output_dir, "git_hash.txt"), "w") as f:
    f.write(git_hash + "\n")

# 5. label_names.json — index-to-class mapping; makes the artifact self-contained
with open(os.path.join(output_dir, "label_names.json"), "w") as f:
    json.dump(
        {"label_names": LABEL_NAMES,
         "id2label": {str(i): n for i, n in enumerate(LABEL_NAMES)},
         "label2id": {n: i for i, n in enumerate(LABEL_NAMES)}},
        f, indent=2,
    )

# 6. Move checkpoint into output_dir
# shutil.move is safe across filesystem/drive boundaries; os.rename raises OSError on Windows
# when source and destination are on different volumes (e.g. temp vs model_registry partition).
final_ckpt = os.path.join(output_dir, f"best_{model_name}_ckpt.pt")
shutil.move(best_ckpt_path, final_ckpt)

print(f"Artifacts saved to: {output_dir}/")
print(f"  config_used.json           — training configuration snapshot")
print(f"  training_log.json          — per-epoch metrics (loss, F1, VRAM, elapsed)")
print(f"  eval_report.json           — per-class precision/recall/F1 on test set")
print(f"  label_names.json           — id2label / label2id mapping for decoding predictions")
print(f"  git_hash.txt               — {git_hash[:12] if len(git_hash) >= 12 else git_hash}...")
print(f"  best_{model_name}_ckpt.pt  — best-epoch model weights")

In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated : {torch.cuda.memory_allocated()/1e6:.1f} MB")
    print(f"GPU memory cached    : {torch.cuda.memory_reserved()/1e6:.1f} MB")
print("Done.")